# Day 052 | Ensemble Methods Deep Dive

## 100 Days of ML for Geotechnical Engineering
**Phase:** Classical Machine Learning

**Author:** Ripon Chandra Malo  
**Date:** YYYY-MM-DD  

---

## What We Will Learn Today

| Concept | What It Means | Geotechnical Example |
|---------|--------------|---------------------|
| VotingClassifier | Combine predictions from multiple classifiers | Multiple experts assess slope stability |
| VotingRegressor | Average predictions from multiple regressors | Blend settlement estimates from different methods |
| Soft vs Hard Voting | Use probabilities vs majority vote | Confidence-weighted vs equal-weight opinions |
| StackingRegressor | Train a meta-model on base model outputs | Learn how to optimally combine expert predictions |
| Blending RF + XGBoost | Combine tree-based ensemble methods | Best of bagging + boosting for soil compressibility |
| Model Diversity | Different models capture different patterns | Each model compensates for others' weaknesses |

---

## Our Geotechnical Problem

We predict **soil compressibility index (Cc)** from lab and field data.  
No single model is perfect — by combining Random Forest (bagging) with Gradient Boosting / XGBoost (boosting),  
we can often achieve better predictions than either model alone.

| Approach | Analogy in Practice |
|----------|--------------------|
| Single model | One engineer's estimate |
| Voting ensemble | Committee average |
| Stacking ensemble | Senior engineer weighting junior opinions |

---

## Part 1: Generate Synthetic Soil Compressibility Data

In [ ]:
# ─────────────────────────────────────────────────────
# IMPORTS AND DATA GENERATION
# ─────────────────────────────────────────────────────

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
n = 400

# Soil properties
liquid_limit = np.random.uniform(25, 80, n)
plasticity_index = liquid_limit * np.random.uniform(0.3, 0.7, n)
void_ratio = 0.4 + 0.015 * liquid_limit + np.random.normal(0, 0.1, n)
water_content = liquid_limit * np.random.uniform(0.5, 1.1, n)
clay_fraction = np.clip(plasticity_index * 1.2 + np.random.normal(0, 8, n), 5, 95)
specific_gravity = np.random.uniform(2.60, 2.80, n)

# Compression index (Cc) — empirical + noise
Cc = (0.009 * (liquid_limit - 10) +
      0.15 * void_ratio +
      0.003 * plasticity_index +
      np.random.normal(0, 0.04, n))
Cc = np.clip(Cc, 0.05, 1.2)

df = pd.DataFrame({
    'liquid_limit': liquid_limit,
    'plasticity_index': plasticity_index,
    'void_ratio': void_ratio,
    'water_content': water_content,
    'clay_fraction': clay_fraction,
    'specific_gravity': specific_gravity,
    'Cc': Cc
})

print(f"Dataset shape: {df.shape}")
print(f"\nTarget (Cc) statistics:")
print(df['Cc'].describe().round(3))

---

## Part 2: Train Individual Base Models

In [ ]:
# ─────────────────────────────────────────────────────
# TRAIN/TEST SPLIT AND BASE MODELS
# ─────────────────────────────────────────────────────

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

features = ['liquid_limit', 'plasticity_index', 'void_ratio',
            'water_content', 'clay_fraction', 'specific_gravity']
X = df[features]
y = df['Cc']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Train individual models
models = {
    'Ridge': Ridge(alpha=1.0),
    'SVR': SVR(kernel='rbf', C=10, epsilon=0.01),
    'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, max_depth=5, random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train_s, y_train)
    y_pred = model.predict(X_test_s)
    results[name] = {
        'R2': r2_score(y_test, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
        'MAE': mean_absolute_error(y_test, y_pred)
    }

results_df = pd.DataFrame(results).T.round(4)
print("Individual Model Performance:")
print("=" * 50)
print(results_df)
print("\nNow let's see if combining them does better!")

---

## Part 3: VotingRegressor — Simple Averaging

In [ ]:
# ─────────────────────────────────────────────────────
# VOTING REGRESSOR — Average multiple model predictions
# ─────────────────────────────────────────────────────

from sklearn.ensemble import VotingRegressor

voting_reg = VotingRegressor(
    estimators=[
        ('ridge', Ridge(alpha=1.0)),
        ('svr', SVR(kernel='rbf', C=10, epsilon=0.01)),
        ('rf', RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)),
        ('gb', GradientBoostingRegressor(n_estimators=200, max_depth=5, random_state=42))
    ]
)

voting_reg.fit(X_train_s, y_train)
y_pred_vote = voting_reg.predict(X_test_s)

r2_vote = r2_score(y_test, y_pred_vote)
rmse_vote = np.sqrt(mean_squared_error(y_test, y_pred_vote))
mae_vote = mean_absolute_error(y_test, y_pred_vote)

print("VotingRegressor (Simple Average):")
print(f"  R2   = {r2_vote:.4f}")
print(f"  RMSE = {rmse_vote:.4f}")
print(f"  MAE  = {mae_vote:.4f}")
print()
best_individual = results_df['R2'].max()
print(f"Best individual R2 = {best_individual:.4f}")
print(f"Voting R2          = {r2_vote:.4f}")
if r2_vote > best_individual:
    print(f"Voting improved by {(r2_vote - best_individual)*100:.2f} percentage points!")
else:
    print("Voting slightly lower — weaker models may dilute the best one.")

---

## Part 4: Weighted VotingRegressor

In [ ]:
# ─────────────────────────────────────────────────────
# WEIGHTED VOTING — Give more weight to better models
# ─────────────────────────────────────────────────────

weights = [results[name]['R2'] for name in ['Ridge', 'SVR', 'Random Forest', 'Gradient Boosting']]
print("Model weights (proportional to R2):")
for name, w in zip(['Ridge', 'SVR', 'RF', 'GB'], weights):
    print(f"  {name:20s}: {w:.4f}")

voting_weighted = VotingRegressor(
    estimators=[
        ('ridge', Ridge(alpha=1.0)),
        ('svr', SVR(kernel='rbf', C=10, epsilon=0.01)),
        ('rf', RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)),
        ('gb', GradientBoostingRegressor(n_estimators=200, max_depth=5, random_state=42))
    ],
    weights=weights
)

voting_weighted.fit(X_train_s, y_train)
y_pred_wvote = voting_weighted.predict(X_test_s)

r2_wvote = r2_score(y_test, y_pred_wvote)
rmse_wvote = np.sqrt(mean_squared_error(y_test, y_pred_wvote))

print(f"\nWeighted Voting R2   = {r2_wvote:.4f}")
print(f"Simple Voting R2     = {r2_vote:.4f}")
print(f"Best Individual R2   = {best_individual:.4f}")

---

## Part 5: VotingClassifier — Soft vs Hard Voting

In [ ]:
# ─────────────────────────────────────────────────────
# VOTING CLASSIFIER — Hard vs Soft voting
# ─────────────────────────────────────────────────────

from sklearn.ensemble import VotingClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# Create classification target: high compressibility (Cc > median)
y_cls = (df['Cc'] > df['Cc'].median()).astype(int)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)
X_train_cs = scaler.fit_transform(X_train_c)
X_test_cs = scaler.transform(X_test_c)

estimators_cls = [
    ('lr', LogisticRegression(max_iter=1000)),
    ('rf', RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42)),
    ('gb', GradientBoostingClassifier(n_estimators=200, max_depth=5, random_state=42))
]

# Hard voting: majority vote
hard_vote = VotingClassifier(estimators=estimators_cls, voting='hard')
hard_vote.fit(X_train_cs, y_train_c)
y_pred_hard = hard_vote.predict(X_test_cs)

# Soft voting: average probabilities
soft_vote = VotingClassifier(estimators=estimators_cls, voting='soft')
soft_vote.fit(X_train_cs, y_train_c)
y_pred_soft = soft_vote.predict(X_test_cs)

print("Hard Voting (majority vote):")
print(f"  Accuracy = {accuracy_score(y_test_c, y_pred_hard):.4f}")
print(f"  F1-Score = {f1_score(y_test_c, y_pred_hard):.4f}")
print()
print("Soft Voting (probability averaging):")
print(f"  Accuracy = {accuracy_score(y_test_c, y_pred_soft):.4f}")
print(f"  F1-Score = {f1_score(y_test_c, y_pred_soft):.4f}")
print()
print("Soft voting usually performs better because it uses confidence info.")

---

## Part 6: StackingRegressor — Learn to Combine

In [ ]:
# ─────────────────────────────────────────────────────
# STACKING REGRESSOR — Meta-learner on top of base models
# ─────────────────────────────────────────────────────

from sklearn.ensemble import StackingRegressor

stacking_reg = StackingRegressor(
    estimators=[
        ('ridge', Ridge(alpha=1.0)),
        ('svr', SVR(kernel='rbf', C=10, epsilon=0.01)),
        ('rf', RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)),
        ('gb', GradientBoostingRegressor(n_estimators=200, max_depth=5, random_state=42))
    ],
    final_estimator=Ridge(alpha=0.5),
    cv=5
)

stacking_reg.fit(X_train_s, y_train)
y_pred_stack = stacking_reg.predict(X_test_s)

r2_stack = r2_score(y_test, y_pred_stack)
rmse_stack = np.sqrt(mean_squared_error(y_test, y_pred_stack))
mae_stack = mean_absolute_error(y_test, y_pred_stack)

print("StackingRegressor Results:")
print(f"  R2   = {r2_stack:.4f}")
print(f"  RMSE = {rmse_stack:.4f}")
print(f"  MAE  = {mae_stack:.4f}")
print()
print("Comparison:")
print(f"  Best Individual   R2 = {best_individual:.4f}")
print(f"  Simple Voting     R2 = {r2_vote:.4f}")
print(f"  Weighted Voting   R2 = {r2_wvote:.4f}")
print(f"  Stacking          R2 = {r2_stack:.4f}")
print()
print("Stacking learns OPTIMAL weights, unlike simple averaging.")

---

## Part 7: RF + XGBoost Stacking for Soil Compressibility

In [ ]:
# ─────────────────────────────────────────────────────
# RF + XGBOOST STACKING — Best of bagging + boosting
# ─────────────────────────────────────────────────────

try:
    from xgboost import XGBRegressor
    has_xgb = True
except ImportError:
    has_xgb = False
    print("XGBoost not installed. Using GradientBoostingRegressor as substitute.")

if has_xgb:
    xgb_model = XGBRegressor(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8, random_state=42
    )
else:
    xgb_model = GradientBoostingRegressor(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        subsample=0.8, random_state=42
    )

rf_xgb_stack = StackingRegressor(
    estimators=[
        ('rf', RandomForestRegressor(n_estimators=300, max_depth=12, random_state=42)),
        ('xgb', xgb_model)
    ],
    final_estimator=Ridge(alpha=1.0),
    cv=5
)

rf_xgb_stack.fit(X_train_s, y_train)
y_pred_rfxgb = rf_xgb_stack.predict(X_test_s)

r2_rfxgb = r2_score(y_test, y_pred_rfxgb)
rmse_rfxgb = np.sqrt(mean_squared_error(y_test, y_pred_rfxgb))

print("RF + XGBoost Stacking:")
print(f"  R2   = {r2_rfxgb:.4f}")
print(f"  RMSE = {rmse_rfxgb:.4f}")
print()
print("Why this works:")
print("  - RF (bagging): reduces variance, captures interactions")
print("  - XGBoost (boosting): reduces bias, learns residual patterns")
print("  - Stacking meta-learner: optimally combines both")

In [ ]:
# ─────────────────────────────────────────────────────
# COMPREHENSIVE COMPARISON PLOT
# ─────────────────────────────────────────────────────

all_results = {
    'Ridge': results['Ridge']['R2'],
    'SVR': results['SVR']['R2'],
    'Random Forest': results['Random Forest']['R2'],
    'Grad. Boosting': results['Gradient Boosting']['R2'],
    'Voting (simple)': r2_vote,
    'Voting (weighted)': r2_wvote,
    'Stacking (all)': r2_stack,
    'RF+XGB Stack': r2_rfxgb
}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart
colors = ['#bdc3c7']*4 + ['#3498db', '#2980b9', '#e74c3c', '#c0392b']
bars = axes[0].barh(list(all_results.keys()), list(all_results.values()),
                     color=colors, edgecolor='black')
axes[0].set_xlabel('R2 Score', fontsize=12)
axes[0].set_title('Individual vs Ensemble Models', fontsize=13, fontweight='bold')
for bar, val in zip(bars, all_results.values()):
    axes[0].text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
                 f'{val:.4f}', va='center', fontsize=10)
axes[0].axvline(x=best_individual, color='gray', linestyle='--', alpha=0.5)

# Scatter: actual vs predicted for best ensemble
axes[1].scatter(y_test, y_pred_rfxgb, alpha=0.6, color='#e74c3c',
                edgecolors='black', s=40, label=f'RF+XGB Stack (R2={r2_rfxgb:.3f})')
min_val = min(y_test.min(), y_pred_rfxgb.min())
max_val = max(y_test.max(), y_pred_rfxgb.max())
axes[1].plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=2, label='Perfect')
axes[1].set_xlabel('Actual Cc', fontsize=12)
axes[1].set_ylabel('Predicted Cc', fontsize=12)
axes[1].set_title('RF+XGB Stacking: Actual vs Predicted', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

---

## Part 8: Understanding How Stacking Works Internally

In [ ]:
# ─────────────────────────────────────────────────────
# HOW STACKING WORKS — Meta-learner coefficients
# ─────────────────────────────────────────────────────

meta_model = rf_xgb_stack.final_estimator_
print("Stacking Meta-Learner Coefficients:")
print(f"  RF weight:      {meta_model.coef_[0]:.4f}")
print(f"  XGBoost weight: {meta_model.coef_[1]:.4f}")
print(f"  Intercept:      {meta_model.intercept_:.4f}")
print()
if abs(meta_model.coef_[0]) > abs(meta_model.coef_[1]):
    print("RF has more influence — bagging captures the main patterns better.")
else:
    print("XGBoost has more influence — boosting captures residual patterns better.")

# Show how predictions are combined
rf_preds = rf_xgb_stack.estimators_[0].predict(X_test_s)
xgb_preds = rf_xgb_stack.estimators_[1].predict(X_test_s)

fig, ax = plt.subplots(figsize=(12, 6))
idx = np.argsort(y_test.values)[:30]
x_pos = np.arange(len(idx))

ax.plot(x_pos, y_test.values[idx], 'ko-', linewidth=2, markersize=6, label='Actual Cc')
ax.plot(x_pos, rf_preds[idx], 's--', color='#2ecc71', alpha=0.7, label='RF prediction')
ax.plot(x_pos, xgb_preds[idx], '^--', color='#e67e22', alpha=0.7, label='XGBoost prediction')
ax.plot(x_pos, y_pred_rfxgb[idx], 'D-', color='#e74c3c', markersize=5, label='Stacked prediction')

ax.set_xlabel('Sample Index (sorted by actual Cc)', fontsize=12)
ax.set_ylabel('Compression Index (Cc)', fontsize=12)
ax.set_title('How Stacking Combines Base Model Predictions', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

---

## Practice Exercises

### Exercise 1: Add More Base Models
Add a KNN regressor and an ElasticNet to the stacking ensemble. Does performance improve or does adding weak models hurt?

### Exercise 2: Try Different Meta-Learners
Replace the Ridge meta-learner with a GradientBoostingRegressor. Compare R2 and check if it overfits.

### Exercise 3: Stacking for Classification
Build a StackingClassifier for the high/low compressibility problem. Compare with the VotingClassifier from Part 5.

In [ ]:
# EXERCISE 1 STARTER:
# from sklearn.neighbors import KNeighborsRegressor
# from sklearn.linear_model import ElasticNet
# extended_stack = StackingRegressor(
#     estimators=[
#         ('rf', RandomForestRegressor(n_estimators=200, random_state=42)),
#         ('gb', GradientBoostingRegressor(n_estimators=200, random_state=42)),
#         ('knn', KNeighborsRegressor(n_neighbors=10)),
#         ('en', ElasticNet(alpha=0.01, l1_ratio=0.5))
#     ],
#     final_estimator=Ridge(alpha=1.0),
#     cv=5
# )
# extended_stack.fit(X_train_s, y_train)
# print(f"Extended stack R2: {r2_score(y_test, extended_stack.predict(X_test_s)):.4f}")

---

## Key Takeaways

1. **VotingRegressor** averages predictions from multiple models. Simple but effective when models have similar strength.

2. **Weighted voting** gives more weight to better-performing models, often outperforming simple averaging.

3. **VotingClassifier** supports both hard (majority vote) and soft (probability average) voting. Soft voting is usually superior.

4. **StackingRegressor** trains a meta-learner that learns optimal combination weights from cross-validated base model outputs.

5. **RF + XGBoost** is a powerful combination: RF reduces variance (bagging), XGBoost reduces bias (boosting). Together they cover each other's weaknesses.

6. In geotechnical engineering, ensemble methods reduce prediction uncertainty — like consulting multiple experts rather than relying on one.

---

## How This Connects to ML

| Today's Concept | ML Equivalent |
|----------------|---------------|
| Committee decision | VotingClassifier/Regressor |
| Weighted expert opinion | Weighted voting ensemble |
| Learning from experts | StackingRegressor meta-learner |
| Bagging + Boosting | RF + XGBoost combination |
| Hard vs Soft voting | Majority vote vs probability averaging |
| Model diversity | Different algorithms capture different patterns |

---

*Day 052 of 100 — The wisdom of the ensemble!*